# 5. `Fan speeds ( <N> <N> <N> <N> <N> <N> )` (switch_module, fan)

In [1]:
from common import *

shape, log = show(5)

log shape 38: switch_module | fan | Fan speeds ( <N> <N> <N> <N> <N> <N> )
21,559 logs (5.2%) from 1 nodes, 2004-09-19 04:42:40 to 2006-04-10 06:03:16
log 50422 at 2005-09-05 14:56:30: 41953 Interconnect-0N00 switch_module fan 1125932190 1 Fan speeds ( 3552 3534 3375 **** 3515 3479 )


In [2]:
print("I'd say that if the speeds were close to zero this should be a concern.")
print("Five of the six are above 1000 RPM (guessing a normal minimum speed here), but the fourth is '****': that fan has no reading.")
print("Should we worry about the fan with no reading?")

I'd say that if the speeds were close to zero this should be a concern.
Five of the six are above 1000 RPM (guessing a normal minimum speed here), but the fourth is '****': that fan has no reading.
Should we worry about the fan with no reading?


Every fan report of the 7 days before, one column per fan

In [3]:
reports = as_frame(logs.search(log.ts - 7 * day, log.ts, log_shape_id=shape.log_shape_id, limit=None))
speeds = reports.message.str.extract(r"\(\s*(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s*\)")
speeds.columns = [f"fan{i}" for i in range(1, 7)]
print(f"{len(reports)} reports, from {sorted(set(reports.node))}")
speeds.tail(5)

6110 reports, from ['Interconnect-0N00']


,fan1,fan2,fan3,fan4,fan5,fan6
6105,3552,3534,3375,****,3515,3479
6106,3552,3534,3375,****,3515,3479
6107,3552,3534,3375,****,3515,3479
6108,3552,3534,3375,****,3515,3479
6109,3552,3534,3375,****,3515,3479


Per fan: reports with no reading, and how many different values it showed

In [4]:
summary = pd.DataFrame({"no_reading": (speeds == "****").sum(), "distinct_values": speeds.nunique()})
display(summary)
assert summary.no_reading.drop("fan4").sum() == 0 and summary.no_reading.fan4 > 0.8 * len(reports)
assert (summary.distinct_values.drop("fan4") == 1).all()

,no_reading,distinct_values
fan1,0,1
fan2,0,1
fan3,0,1
fan4,5372,66
fan5,0,1
fan6,0,1


Maybe the 4th fan isn't working. I'd say we should alert the user with low severity.

Just to go the extra mile, we should check whether this affects the temperature of this device, has it overheated?

We dont know what temperature warnings look like so let's examine all logs on this component in the past day.

In [5]:
logs.node_timeline(log.node, log.ts - day, log.ts)

[TimelineEntry(component='switch_module', state='fan', template='Fan speeds ( <N> <N> <N> <N> <N> <N> )', log_shape_id=38, first_ts='2005-09-04 14:56:58', last_ts='2005-09-05 14:56:30', count=2833)]

Only fan reports there, so the day does not show what a heat warning looks like.
- Every log shape that reports heat, as known at the time of this log (`past` sees nothing later than the log)

In [6]:
past = LogTools(conn, visible=log.ts)
heat = pd.concat([past.find_log_shapes(["temperature", "ambient", "threshold"]), past.find_log_shapes(state="temperature")])
heat = heat.drop_duplicates("log_shape_id", ignore_index=True)
heat

,log_shape_id,component,state,template,n_logs,n_nodes,first_seen,last_seen,share
0,99,node,temperature,ambient=<N>,29989,257,2003-12-26 14:58:30,2005-09-05 13:40:30,0.096
1,28,switch_module,temphigh,Temperature (<N>C) exceeds warning threshold,590,8,2005-03-26 09:37:48,2005-03-26 11:01:01,0.002
2,24,node,psu,psu failure\ ambient=<N>,192,59,2004-01-03 08:02:30,2005-08-31 10:30:00,0.001
3,27,node,temphigh,ambient=<N> threshold exceeded,23,13,2004-06-02 16:59:30,2005-07-25 22:24:31,0.000
4,39,node,fan,fan failure\ ambient=<N>,2,1,2004-10-21 20:18:30,2004-10-21 21:56:34,0.000
5,96,gige,temperature,warning,38117,8,2003-12-26 12:36:12,2005-09-05 14:51:27,0.122
6,98,gige,temperature,normal,35434,7,2003-12-26 12:41:12,2005-09-05 14:47:09,0.113
7,97,gige,temperature,critical,3070,7,2003-12-26 12:38:45,2005-08-10 20:36:36,0.010
8,100,node,temperature,,1,1,2005-07-02 14:50:31,2005-07-02 14:50:31,0.000


One of them comes from switches: `Temperature (<N>C) exceeds warning threshold`.
- The last day it was logged before this log, per switch

In [7]:
switch_hot = heat[heat.component == "switch_module"].iloc[0]
last_hot = datetime.fromisoformat(switch_hot.last_seen)
print(f"last logged at {last_hot}, {(log.ts - last_hot).days} days before this log, {switch_hot.n_logs} times in all, first at {switch_hot.first_seen}")
hot_switches = dict(past.count_logs("node", last_hot - day, last_hot, log_shape_id=int(switch_hot.log_shape_id)))
display(pd.Series(hot_switches, name="n_logs"))
assert log.node not in hot_switches and sum(hot_switches.values()) == switch_hot.n_logs

last logged at 2005-03-26 11:01:01, 163 days before this log, 590 times in all, first at 2005-03-26 09:37:48


Interconnect-0N03    170
Interconnect-0N02    110
Interconnect-1N01     97
Interconnect-1N03     57
Interconnect-0N01     51
Interconnect-1T01     48
Interconnect-1N02     35
Interconnect-1T03     22
Name: n_logs, dtype: int64

All of them on one morning, and none from this switch. Did fan 4 already have no reading back then?
- Fan reports and reports with no reading, per 30 days, going back from this log

In [8]:
windows = []
for back in range(7):
    end = log.ts - back * 30 * day
    all_reports = past.stats(end - 30 * day, end, log_shape_id=shape.log_shape_id)
    no_reading = past.stats(end - 30 * day, end, log_shape_id=shape.log_shape_id, text="****")
    windows.append({"from": end - 30 * day, "to": end, "reports": all_reports.n_logs, "no_reading": no_reading.n_logs, "first_no_reading": no_reading.first_seen})
windows = pd.DataFrame(windows)
display(windows)
first_no_reading = datetime.fromisoformat(windows.first_no_reading.dropna().min())
print(f"first report with no reading in these 7 months: {first_no_reading}, {(first_no_reading - last_hot).days} days after the last temperature warning")
assert first_no_reading > last_hot

,from,to,reports,no_reading,first_no_reading
0,2005-08-06 14:56:30,2005-09-05 14:56:30,6462,5689,2005-08-06 14:58:29
1,2005-07-07 14:56:30,2005-08-06 14:56:30,1508,1294,2005-07-09 17:02:48
2,2005-06-07 14:56:30,2005-07-07 14:56:30,510,443,2005-06-20 12:14:29
3,2005-05-08 14:56:30,2005-06-07 14:56:30,4,0,None
4,2005-04-08 14:56:30,2005-05-08 14:56:30,0,0,None
5,2005-03-09 14:56:30,2005-04-08 14:56:30,0,0,None
6,2005-02-07 14:56:30,2005-03-09 14:56:30,0,0,None


first report with no reading in these 7 months: 2005-06-20 12:14:29, 86 days after the last temperature warning


Since fan 4 has had no reading, did this switch log anything about heat, or anything else?
- Everything this switch logged in the 30 days before

In [9]:
since = past.shapes_between(log.ts - 30 * day, log.ts, node=log.node)
display(since)
# no switch logged a temperature warning after last_hot, so none came since fan 4 first had no reading
assert "temphigh" not in set(since.state) and last_hot < first_no_reading

,component,state,template,n_logs,n_sources,first,last
0,switch_module,fan,Fan speeds ( <N> <N> <N> <N> <N> <N> ),6462,1,2005-08-06 14:58:29,2005-09-05 14:56:30
1,switch_module,error,link errors remain current,7,1,2005-08-10 21:18:13,2005-08-25 19:54:10


# Verdict: low severity, a hardware ticket. Fan 4 of this switch, or its sensor, has no reading, and nothing shows the switch running hot.
- Fan 4 has no reading in most reports of the last three months, and the other five fans never change.
- The only heat message a switch writes is `Temperature (<N>C) exceeds warning threshold`. Before this log it was written on one morning only, more than five months earlier, by eight other switches and never by this one. The first fan report with no reading came 86 days after that morning, and no switch has logged a temperature warning since.
- In the 30 days before this log the switch wrote fan reports and a few `link errors remain current`, and nothing about heat.
- So: open a ticket to check fan 4 and its sensor at the next maintenance. Not urgent. It becomes urgent if this switch logs a temperature warning.

This is the only switch that sends this log shape (fan speed reports). The shape cannot tell this: six plain numbers would be the same shape. The value has to be read.